# Exercise 10, advection and diffusion in a river

These exercises go with **Lecture 10 &mdash; Advection and diffusion in a river**. They
reuse the `route` solver, the `transport` solver, the channel geometry and the analytic
plume from that notebook, all reproduced in the setup cell.

Fill only the cells marked

```python
# ==== YOUR CODE ====
```


In [ ]:
# Google Colab setup: installs the packages this notebook uses (runs only on Colab).
import sys
if "google.colab" in sys.modules:
    %pip install -q numpy pandas matplotlib scipy xarray netcdf4 pooch


## Setup (given &mdash; just run it)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- channel geometry -> celerity c and diffusivity D (Lecture 10) ---
S0, B, n_m, Qref = 3e-4, 50.0, 0.040, 150.0
h = (Qref * n_m / (B * S0 ** 0.5)) ** 0.6
V = Qref / (B * h)
c = (5.0 / 3.0) * V
D = Qref / (2.0 * B * S0)
print(f"h = {h:.2f} m   V = {V:.2f} m/s   c = {c:.2f} m/s   D = {D:.0f} m2/s")


def route(inflow, dt, *, celerity, diffusivity, dx, reach_length,
          q_init=None, return_field=False):
    """Linear diffusive-wave routing: dQ/dt + c dQ/dx = D d2Q/dx2 (Lecture 10)."""
    cc, Dd = float(celerity), float(diffusivity)
    n_x = int(round(reach_length / dx)) + 1
    dt_stable = 0.9 / (cc / dx + 2.0 * Dd / dx ** 2)
    n_sub = max(1, int(np.ceil(dt / dt_stable)))
    hh = dt / n_sub
    Cr, Ne = cc * hh / dx, Dd * hh / dx ** 2
    inflow = np.asarray(inflow, float)
    Q = np.full(n_x, inflow[0] if q_init is None else float(q_init))
    out = np.empty(len(inflow))
    field = np.empty((len(inflow), n_x)) if return_field else None
    for k in range(len(inflow)):
        q_prev = inflow[k - 1] if k > 0 else inflow[0]
        for m in range(n_sub):
            Q[0] = q_prev + (inflow[k] - q_prev) * (m + 1) / n_sub
            adv = -Cr * (Q[1:-1] - Q[:-2])
            dif = Ne * (Q[2:] - 2.0 * Q[1:-1] + Q[:-2])
            Q[1:-1] = Q[1:-1] + adv + dif
            Q[-1] = Q[-2]
        out[k] = Q[-1]
        if return_field:
            field[k] = Q
    return (out, field) if return_field else out


def transport(C0, dt, *, velocity, dispersion, dx, n_steps, inflow_conc=None):
    """Advection-dispersion of a concentration profile (centred advection, Lecture 10).
    inflow_conc: optional array of length n_steps giving C at x=0 each step."""
    Vv, DL = float(velocity), float(dispersion)
    Cr, Ne = Vv * dt / dx, DL * dt / dx ** 2
    C = np.asarray(C0, float).copy()
    hist = [C.copy()]
    for k in range(n_steps):
        adv = -Cr * (C[2:] - C[:-2]) / 2.0
        dif = Ne * (C[2:] - 2.0 * C[1:-1] + C[:-2])
        C[1:-1] = C[1:-1] + adv + dif
        C[0] = 0.0 if inflow_conc is None else inflow_conc[k]
        C[-1] = C[-2]
        hist.append(C.copy())
    return hist


def flood_pulse(t_h, t_peak=4.0, shape=6.0, base=20.0, peak=180.0):
    tau = np.maximum(t_h, 1e-9) / t_peak
    g = tau ** shape * np.exp(shape * (1.0 - tau))
    return base + (peak - base) * g


def plume_analytic(x, t, M, A, Vv, DL):
    return M / (A * np.sqrt(4 * np.pi * DL * t)) * np.exp(-(x - Vv * t) ** 2 / (4 * DL * t))


df = pd.read_csv("data/chattooga_daily.csv", comment="#",
                 parse_dates=["date"], index_col="date")
df["Q_cms"] = df["q_mm"] * 536.0 / 86.4
print("setup ready.")


## Exercise 1 &mdash; a discharge-dependent celerity, and wave steepening

The linear model uses one celerity $c$ for the whole wave. In reality
$c = \tfrac{5}{3}V$ *grows with discharge*, so the crest travels faster than the base
and the rising limb steepens as the wave moves downstream &mdash; the start of a
**kinematic shock**.

**Background.** For a wide channel with Manning friction,

$$V = \left(\frac{Q\,n}{B\,\sqrt{S_0}}\right)^{0.6}\Big/ \dots \quad\Longrightarrow\quad
V(Q) = \frac{1}{h(Q)}\,\frac{Q}{B},\qquad h(Q) = \left(\frac{Q\,n}{B\sqrt{S_0}}\right)^{0.6},$$

and $c(Q) = \tfrac{5}{3}V(Q)$. You will write a **non-linear** router that recomputes
$c$ from the *local* discharge every sub-step.

**Your task.**
1. Complete `celerity_of_Q(Q)` returning $c$ for a discharge `Q` (scalar or array).
2. Complete the marked lines in `route_nonlinear` so the advection coefficient uses the
   local celerity `c_local = celerity_of_Q(Q[:-1])` (the cell the water comes from).
3. Route the synthetic flood with both routers and compare (plot given).

**Hints.**
* `h = (Q * n_m / (B * np.sqrt(S0)))**0.6`, then `V = Q / (B * h)`, then `c = 5/3 * V`.
* Guard against `Q = 0`: use `np.maximum(Q, 1.0)` inside `celerity_of_Q`.
* In the loop, the upwind term is `-(c_local * dt_sub / dx) * (Q[1:-1] - Q[:-2])`.
  Keep the diffusion term linear (constant `D`) for simplicity.


In [ ]:
def celerity_of_Q(Q):
    """Kinematic wave celerity c = 5/3 V for discharge Q [m3/s] (Manning, wide channel)."""
    Q = np.maximum(np.asarray(Q, float), 1.0)
    # ==== YOUR CODE ====
    # hh = (Q * n_m / (B * np.sqrt(S0)))**0.6
    # VV = Q / (B * hh)
    # return 5/3 * VV
    return np.full_like(Q, c)        # <-- replace (this line just returns the constant c)
    # ===================


def route_nonlinear(inflow, dt, *, diffusivity, dx, reach_length):
    n_x = int(round(reach_length / dx)) + 1
    # fixed sub-step from the *fastest* celerity we expect (crest of the inflow)
    c_max = float(celerity_of_Q(inflow.max()))
    dt_sub = 0.9 / (c_max / dx + 2.0 * diffusivity / dx ** 2)
    n_sub = max(1, int(np.ceil(dt / dt_sub)))
    hsub = dt / n_sub
    Ne = diffusivity * hsub / dx ** 2
    Q = np.full(n_x, inflow[0]); out = np.empty(len(inflow))
    for k in range(len(inflow)):
        q_prev = inflow[k - 1] if k > 0 else inflow[0]
        for m in range(n_sub):
            Q[0] = q_prev + (inflow[k] - q_prev) * (m + 1) / n_sub
            c_local = celerity_of_Q(Q[1:-1])            # celerity in each interior cell
            # ==== YOUR CODE ====
            # adv = -(c_local * hsub / dx) * (Q[1:-1] - Q[:-2])
            adv = np.zeros(n_x - 2)                     # <-- replace
            # ===================
            dif = Ne * (Q[2:] - 2.0 * Q[1:-1] + Q[:-2])
            Q[1:-1] = Q[1:-1] + adv + dif
            Q[-1] = Q[-2]
        out[k] = Q[-1]
    return out


dt = 60.0
t_h = np.arange(0, 96, dt / 3600)
inflow = flood_pulse(t_h)
dx, reach = 500.0, 60_000.0

q_linear = route(inflow, dt, celerity=c, diffusivity=D, dx=dx, reach_length=reach)
q_nonlin = route_nonlinear(inflow, dt, diffusivity=D, dx=dx, reach_length=reach)
print(f"linear    outflow peak {q_linear.max():.0f} m3/s at t = {t_h[q_linear.argmax()]:.1f} h")
print(f"nonlinear outflow peak {q_nonlin.max():.0f} m3/s at t = {t_h[q_nonlin.argmax()]:.1f} h")


In [ ]:
# --- plot (given) ---
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_h, inflow, "k", lw=2, label="inflow")
ax.plot(t_h, q_linear, lw=1.5, label="linear routing (constant c)")
ax.plot(t_h, q_nonlin, lw=1.5, label="non-linear routing (c grows with Q)")
ax.set_xlabel("time [h]"); ax.set_ylabel("discharge [m$^3$ s$^{-1}$]")
ax.set_title(f"Outflow at {reach/1000:.0f} km")
ax.legend(); ax.grid(alpha=0.3); ax.set_xlim(0, 48)
fig.tight_layout()


```{admonition} Answers
:class: note
1. Print `celerity_of_Q(20)` and `celerity_of_Q(180)`. How much faster does the crest
   travel than the baseflow ahead of it?
2. Compare the **rising limb** of the non-linear outflow with the linear one. Is it
   steeper or gentler, and why does $c(Q)$ do that? What would happen to the front over
   a reach several times longer (a kinematic shock)?
3. The non-linear peak does not simply arrive earlier &mdash; most of the wave is at a
   discharge well below the 150 m³ s⁻¹ used to linearise, so it travels *slower* on
   average, even though the crest itself is quick. Explain how both can be true at once.
4. Real flood-routing schemes (Muskingum&ndash;Cunge) update $c$ and $D$ from the local
   flow every step, exactly as you did here. Why is that worth the extra cost for a
   large flood?
```


## Exercise 2 &mdash; two reaches and a tributary

Real rivers are networks. Here you route the Chattooga flood through a first reach, add
a tributary at its confluence, and route the combined flow through a second reach.

**Your task.**
1. Route the observed Chattooga flood (given as `main_inflow`, daily) through
   **Reach 1** (30 km).
2. Build a tributary hydrograph: a scaled, time-shifted copy of the main inflow
   (`trib = 0.5 * shift(main_inflow, lag_days)`), add it to the Reach-1 outflow.
3. Route the sum through **Reach 2** (30 km). Compare the outlet peak for
   `lag_days = -1, 0, +1` (tributary peaks a day early / together / a day late).

**Hints.**
* `np.roll(a, k)` shifts an array by `k` (positive = later). Zero the wrapped end.
* Route daily series directly: `route(series, 86400.0, celerity=c, diffusivity=D,
  dx=3000.0, reach_length=30_000.0)`.
* The confluence inflow to Reach 2 is `reach1_out + trib` (both daily, same length).


In [ ]:
peak_day = df["Q_cms"].idxmax()
event = df.loc[peak_day - pd.Timedelta(days=15): peak_day + pd.Timedelta(days=20), "Q_cms"]
main_inflow = event.to_numpy()
day = np.arange(len(main_inflow))

def route_daily(series, reach_km):
    return route(series, 86400.0, celerity=c, diffusivity=D,
                 dx=3000.0, reach_length=reach_km * 1000)

# ==== YOUR CODE ====
reach1_out = None                    # <-- route main_inflow through 30 km

outlet_peak = {}
for lag_days in (-1, 0, 1):
    # trib = 0.5 * np.roll(main_inflow, lag_days); zero the wrapped values
    # confluence = reach1_out + trib
    # reach2_out = route_daily(confluence, 30)
    # outlet_peak[lag_days] = reach2_out.max()
    outlet_peak[lag_days] = np.nan   # <-- replace
# ===================

print("outlet peak [m3/s] vs tributary timing:")
for k, v in outlet_peak.items():
    print(f"  tributary lag {k:+d} day : {v}")


In [ ]:
# --- plot (given) ---
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(day, main_inflow, "k", lw=1.5, label="Chattooga inflow (Reach 1 top)")
if reach1_out is not None:
    ax.plot(day, reach1_out, lw=1.2, label="after Reach 1")
    for lag_days, style in zip((-1, 0, 1), ("-", "--", ":")):
        trib = 0.5 * np.roll(main_inflow, lag_days)
        if lag_days > 0: trib[:lag_days] = 0
        if lag_days < 0: trib[lag_days:] = 0
        r2 = route_daily(reach1_out + trib, 30)
        ax.plot(day, r2, style, lw=1.2, label=f"outlet, tributary lag {lag_days:+d} d")
ax.set_xlabel("day"); ax.set_ylabel("discharge [m$^3$ s$^{-1}$]")
ax.set_title("Two reaches with a tributary confluence")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout()


```{admonition} Answers
:class: note
1. Which tributary timing gives the **highest** outlet peak, and why? (This is why flood
   forecasters care about the *relative* timing of sub-catchment peaks, not just their
   size.)
2. Routing through Reach 1 first *delayed and spread* the main flood. Does that make the
   confluence more or less dangerous when the tributary is unrouted (fresh from a flashy
   side catchment)?
3. Sketch how you would extend this to a 4-tributary network. What does each channel
   segment need?
```


## Exercise 3 &mdash; a continuous spill (Ogata&ndash;Banks)

Lecture 10 released the pollutant all at once. Here a pipe leaks a **constant
concentration** $C_0$ into the stream from $t = 0$ onward. The exact solution is the
**Ogata&ndash;Banks** equation:

$$\frac{C(x,t)}{C_0} = \tfrac12\,\mathrm{erfc}\!\left(\frac{x - Vt}{2\sqrt{D_L t}}\right)
   + \tfrac12\,e^{\,Vx/D_L}\,\mathrm{erfc}\!\left(\frac{x + Vt}{2\sqrt{D_L t}}\right).$$

The second term is tiny unless $D_L$ is large; you may keep both.

**Your task.**
1. Complete `ogata_banks(x, t, V, DL)` returning $C/C_0$ (use `scipy.special.erfc`).
2. Run the given numerical `transport` solver with a **constant inflow concentration**
   and overlay the analytic curve at a fixed downstream point.
3. Find the time for the intake at $x = 12$ km to reach $C = 0.5\,C_0$.

**Hints.**
* `from scipy.special import erfc`.
* The exponential term can overflow; compute it as
  `np.exp(np.minimum(V * x / DL, 300))` or just clip.
* For the "time to reach $0.5 C_0$", evaluate `ogata_banks(12_000, t_array, V, DL)` and
  find the first `t` where it crosses 0.5 (`np.argmax(curve > 0.5)`).


In [ ]:
from scipy.special import erfc

DL = 20.0          # longitudinal dispersion [m2/s]

def ogata_banks(x, t, V, DL):
    """C / C0 for a continuous release starting at t = 0 (x, t may be arrays)."""
    x = np.asarray(x, float); t = np.asarray(t, float)
    # ==== YOUR CODE ====
    # term1 = 0.5 * erfc((x - V*t) / (2*np.sqrt(DL*t)))
    # term2 = 0.5 * np.exp(np.minimum(V*x/DL, 300)) * erfc((x + V*t) / (2*np.sqrt(DL*t)))
    # return term1 + term2
    return np.zeros_like(x + t)      # <-- replace
    # ===================


# numerical solver with a constant upstream concentration (given)
dx_s, dt_s = 20.0, 8.0
xg = np.arange(0, 25_000 + dx_s, dx_s)
n_steps = int(6 * 3600 / dt_s)
C0 = 1.0
hist = transport(np.zeros_like(xg), dt_s, velocity=V, dispersion=DL, dx=dx_s,
                 n_steps=n_steps, inflow_conc=np.full(n_steps, C0))

x_probe = 12_000.0
t_probe = np.arange(dt_s, 6 * 3600, 60.0)
analytic_probe = ogata_banks(x_probe, t_probe, V, DL)

# ==== YOUR CODE ====
# t_half : first time (s) at which analytic_probe crosses 0.5
t_half = None                       # <-- replace with t_probe[np.argmax(analytic_probe > 0.5)]
# ===================
print(f"advective travel time x/V = {x_probe / V / 3600:.2f} h")
print(f"time to reach 0.5 C0 at the intake: {t_half}")


In [ ]:
# --- plot (given) ---
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for hr in [1, 2, 3, 4, 5]:
    k = int(hr * 3600 / dt_s)
    ax[0].plot(xg / 1000, hist[k], lw=1.5, label=f"t = {hr} h (numeric)")
    ax[0].plot(xg / 1000, ogata_banks(xg, hr * 3600, V, DL), "k:", lw=1.0)
ax[0].set_xlabel("distance downstream [km]"); ax[0].set_ylabel("$C / C_0$")
ax[0].set_title("advancing front: numeric (colour) vs Ogata-Banks (dotted)")
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)

ax[1].plot(t_probe / 3600, analytic_probe, lw=1.8)
ax[1].axhline(0.5, color="r", ls="--", lw=1)
if t_half is not None:
    ax[1].axvline(t_half / 3600, color="r", lw=1)
ax[1].set_xlabel("time since spill started [h]"); ax[1].set_ylabel("$C / C_0$ at 12 km")
ax[1].set_title("breakthrough at the intake"); ax[1].grid(alpha=0.3)
fig.tight_layout()


```{admonition} Answers
:class: note
1. How does the time to reach $0.5\,C_0$ compare with the pure advective travel time
   $x/V$? Dispersion spreads the front both upstream and downstream of $x = Vt$
   &mdash; which way does that nudge the half-concentration time, and by how much here?
2. For a continuous release the intake concentration keeps rising toward $C_0$ &mdash;
   it never clears while the leak continues. Contrast this with the instantaneous spill
   in Lecture 10, where the plume passed and the water cleared.
3. Increase `DL` to 100 m² s⁻¹. Does the front arrive earlier, and does it get sharper
   or more smeared?
```


## Exercise 4 &mdash; calibrate the reach from an upstream and a downstream gauge

In practice you do not know $c$ and $D$ for a real reach &mdash; you infer them by
routing an upstream gauge and matching a downstream gauge, exactly the calibration idea
of Lecture 8.

**Your task.**
1. A synthetic "downstream gauge" is given: the Chattooga flood routed with *true*
   parameters $c^\* , D^\*$ (hidden) plus measurement noise.
2. Grid-search $c$ and $D$: for each pair, route the upstream series and compute the
   RMSE against the downstream gauge.
3. Report the best $(c, D)$ and compare with the values from the channel geometry.

**Hints.**
* `route(upstream, 86400.0, celerity=c_try, diffusivity=D_try, dx=3000.0,
  reach_length=45_000.0)`.
* `rmse = np.sqrt(np.mean((routed[mask] - downstream[mask])**2))`, `mask` skipping the
  first few spin-up days.
* `np.unravel_index(np.argmin(err), err.shape)` gives the best grid cell.


In [ ]:
rng = np.random.default_rng(7)
peak_day = df["Q_cms"].idxmax()
ev = df.loc[peak_day - pd.Timedelta(days=20): peak_day + pd.Timedelta(days=30), "Q_cms"]
upstream = ev.to_numpy()

c_true, D_true = 1.35, 6500.0                       # hidden "truth"
clean = route(upstream, 86400.0, celerity=c_true, diffusivity=D_true,
              dx=3000.0, reach_length=45_000.0)
downstream = clean * (1 + 0.03 * rng.standard_normal(len(clean)))   # 3% measurement noise

c_grid = np.linspace(0.8, 2.2, 19)
D_grid = np.linspace(2000, 12000, 19)
spinup = np.arange(len(upstream)) >= 5


In [ ]:
# ==== YOUR CODE ====
err = np.full((len(c_grid), len(D_grid)), np.nan)
for i, c_try in enumerate(c_grid):
    for j, D_try in enumerate(D_grid):
        # routed = route(upstream, 86400.0, celerity=c_try, diffusivity=D_try,
        #                dx=3000.0, reach_length=45_000.0)
        # err[i, j] = np.sqrt(np.mean((routed[spinup] - downstream[spinup])**2))
        pass

# best-fit indices and values
# bi, bj = np.unravel_index(np.nanargmin(err), err.shape)
c_best, D_best = np.nan, np.nan     # <-- replace with c_grid[bi], D_grid[bj]
# ===================

print(f"true      c = {c_true:.2f} m/s   D = {D_true:.0f} m2/s")
print(f"recovered c = {c_best}   D = {D_best}")
print(f"geometry  c = {c:.2f} m/s   D = {D:.0f} m2/s")


In [ ]:
# --- plot (given) ---
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
im = ax[0].pcolormesh(D_grid, c_grid, err, shading="auto", cmap="viridis")
ax[0].plot(D_true, c_true, "r*", ms=14, label="truth")
if np.isfinite(D_best):
    ax[0].plot(D_best, c_best, "wo", ms=7, label="best fit")
ax[0].set_xlabel("D [m$^2$/s]"); ax[0].set_ylabel("c [m/s]")
ax[0].set_title("RMSE surface"); ax[0].legend(fontsize=8)
fig.colorbar(im, ax=ax[0], label="RMSE [m$^3$/s]")

d = np.arange(len(upstream))
ax[1].plot(d, upstream, "k", lw=1, label="upstream gauge")
ax[1].plot(d, downstream, "o", ms=3, color="tab:orange", label="downstream gauge (noisy)")
if np.isfinite(c_best):
    fit = route(upstream, 86400.0, celerity=c_best, diffusivity=D_best,
                dx=3000.0, reach_length=45_000.0)
    ax[1].plot(d, fit, "tab:red", lw=1.5, label="routed with best fit")
ax[1].set_xlabel("day"); ax[1].set_ylabel("discharge [m$^3$ s$^{-1}$]")
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
fig.tight_layout()


```{admonition} Answers
:class: note
1. How close is the recovered $(c, D)$ to the truth? Which of the two is better
   constrained by the data, and why? (Look at the shape of the RMSE surface.)
2. Do the geometry-based $c$ and $D$ from the setup cell fall inside the low-RMSE
   region? If not, what does that tell you about estimating channel parameters from a
   handbook Manning's $n$?
3. This used a clean synthetic "truth". List two reasons a real downstream gauge would
   be harder to match even with the right $c$ and $D$ (recall Lecture 10 Section 4 and
   Lecture 8 Section 7).
```


## Where this goes next

* Non-linear routing (Exercise 1) and network routing (Exercise 2) are what HEC-HMS and
  the routing modules of SWAT and VIC do on every river segment.
* Calibrating transport parameters (Exercises 3&ndash;4) is the basis of tracer studies
  and water-quality modelling &mdash; part of **Module 5**.

```{note} Sources
Original teaching material for **CE524 Applied Hydroclimatology**; reuses the solvers
and Chattooga data of Lecture 10. Text under CC BY-SA 4.0; see
[CREDITS.md](https://github.com/drvivekhydro/hydroclimatology/blob/main/CREDITS.md).
```
